# Example 1.2.1 / 2: Restrictions That Use Other Data

Companion notebook for **1.2.1 Restrictions, Defaults and Checks** in the AMPL lab Documentation, section "Restrictions That Use Other Data". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Restrictions That Use Other Data

The expression in a restriction may use sets and parameters declared earlier, and the
declaration's own index. Then each value can have a different limit. When an indexed
parameter gets its values from a dictionary, amplpy tests them as it assigns them; here the
assignment in `data.py` raises a `RuntimeError` for `f_max["CHK"]`, so `run.py` never runs:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set FOOD;
param f_min {FOOD} >= 0;
param f_max {j in FOOD} >= f_min[j];   # each maximum at least its own minimum

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.read("model.mod")

In [4]:
# This step fails on purpose, as on the page; the error is printed so the notebook goes on
try:
    # data.py
    ampl.set["FOOD"] = ["BEEF", "CHK", "FISH"]
    ampl.param["f_min"] = {"BEEF": 2, "CHK": 2, "FISH": 2}
    ampl.param["f_max"] = {"BEEF": 10, "CHK": 1, "FISH": 10}
except Exception as error:
    print(f"{type(error).__name__}: {error}")

RuntimeError: Error executing "let" command:
error processing param f_max['CHK']:
	failed check: param f_max['CHK'] = 1
		is not >= 2
		-- not >= f_min[j];


The rest of this run is left out: on the page it stops at the error above.

### Pure AMPL: data.dat and model.run

The exception doesn't undo the assignment: if the program catches it and goes on,
`f_max["CHK"]` is 1 and `to_pandas()` returns it without complaint. Treat the exception as
the end of that data set. The test isn't always at the assignment, either: with the bad value
on the last product, `FISH`, the assignment succeeds and the error comes from the first
`to_pandas()` that uses it.

The same in pure AMPL, with the same `model.mod`:

In [5]:
%%writefile data.dat
set FOOD := BEEF CHK FISH;
param:  f_min  f_max :=
  BEEF    2     10
  CHK     2      1
  FISH    2     10 ;

Writing data.dat


In [6]:
%%ampl_eval
reset;
model model.mod;
data data.dat;

In [7]:
# This step fails on purpose, as on the page; the error is printed so the notebook goes on
try:
    ampl.eval(r"""
# model.run
display f_max;
""")
except Exception:
    pass  # AMPL has printed the error above

[ERROR] 
	Error executing "display" command:
	error processing param f_max['CHK']:
		failed check: param f_max['CHK'] = 1
			is not >= 2
			-- not >= f_min[j];


f_max [*] :=
BEEF  10
 CHK   1
FISH  10
;

